# How to use a DIAL interceptor

An [interceptor](https://github.com/epam/ai-dial-interceptors-sdk) is middleware that DIAL Core runs over requests and responses travelling between a caller and a model. This notebook calls a model that has a PII-redacting interceptor attached, and shows that the model never receives the original personal data.

The interceptor is the worked example from `dial-samples/pii-interceptor`. It replaces email addresses, phone numbers and Social Security numbers with placeholders on the way in, and restores them on the way out.

In [ ]:
!pip install -q requests==2.32.3

In [ ]:
import os
import requests

**Step 1**: point `DIAL_URL` at a running DIAL Core. `protected-model` is a stand-in model with the interceptor attached, so no provider API key is needed.

In [ ]:
DIAL_URL = os.environ.get("DIAL_URL", "http://localhost:8080")
APP_NAME = "protected-model"
API_KEY = "dial_api_key"

**Step 2**: send a message containing PII.

In [ ]:
EMAIL = "john.doe@example.com"
PHONE = "555-123-4567"
PROMPT = f"My email is {EMAIL} and my phone is {PHONE}."

response = requests.post(
    f"{DIAL_URL}/openai/deployments/{APP_NAME}/chat/completions",
    headers={"Api-Key": API_KEY, "Content-Type": "application/json"},
    json={"messages": [{"role": "user", "content": PROMPT}]},
    timeout=60,
)
response.raise_for_status()

content = response.json()["choices"][0]["message"]["content"]
print(content)

The reply has two parts. The first is what reaches you — the interceptor has put your original email and phone back. The second is what the model actually received, with the placeholders shown in guillemets so the interceptor does not restore them on the way out.

**Step 3**: verify the redaction actually happened.

A round trip on its own proves nothing: the caller sees the same text whether or not the interceptor did any work. What matters is what the *model* received, so assert on that.

In [ ]:
SEPARATOR = "--- the model received ---"

assert SEPARATOR in content, "the stand-in model did not report what it received"
restored, seen = content.split(SEPARATOR, 1)

# The model must never have seen the real values.
assert EMAIL not in seen, "the model received the original email address"
assert PHONE not in seen, "the model received the original phone number"
assert "«EMAIL_" in seen, "the model did not receive an EMAIL placeholder"
assert "«PHONE_" in seen, "the model did not receive a PHONE placeholder"

# The caller must get the original values back.
assert EMAIL in restored, "the email was not restored in the response"
assert PHONE in restored, "the phone number was not restored in the response"

print("PII redacted before the model, restored for the caller")

**Step 4**: compare with a model that has no interceptor.

`echo` repeats your message back without any interception, so the PII travels through untouched. This is the contrast the interceptor exists to remove.

In [ ]:
unprotected = requests.post(
    f"{DIAL_URL}/openai/deployments/echo/chat/completions",
    headers={"Api-Key": API_KEY, "Content-Type": "application/json"},
    json={"messages": [{"role": "user", "content": PROMPT}]},
    timeout=60,
)
unprotected.raise_for_status()

echoed = unprotected.json()["choices"][0]["message"]["content"]
assert EMAIL in echoed, "expected the un-intercepted application to see the raw email"
print(echoed)

## Next steps

- The interceptor source, and a standalone Docker Compose stack you can run on its own, are in [`dial-samples/pii-interceptor`](https://github.com/epam/ai-dial/tree/main/dial-samples/pii-interceptor).
- To build one yourself, follow [Tutorial: PII-redacting interceptor](https://docs.dialx.ai/v2/building-with-dial/interceptors/tutorial-pii-interceptor).
- To attach an interceptor to your own deployments, see [Configure and assign interceptors](https://docs.dialx.ai/v2/building-with-dial/interceptors/configuration-and-assignment).